# 이커머스 멀티 모달 데이터 분석 및 RFM 리포트

UCI Online Retail 거래 데이터를 기반으로 수치형 거래 정보, 상품명 텍스트, 상품 코드/상품명에서 생성한 NumPy 이미지 배열 피처를 하나의 분석 흐름으로 처리한다.

In [ ]:
from pathlib import Path
from src.pipeline import DataAnalyzer

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
analyzer = DataAnalyzer(PROJECT_ROOT / 'data/ecommerce/data.csv')
df = analyzer.load_data()
df.info()
df.head()


데이터는 541,909건, 17개 컬럼으로 구성된다. 거래 기간은 2010-12-01부터 2011-12-09까지이며, RFM 계산에 사용된 고객 수는 4,338명이다. 단가 평균은 4.61, 중앙값은 2.08로 고가 상품과 비정상 입력값이 평균을 끌어올리는 오른쪽 꼬리 분포가 확인된다.

In [ ]:
df.describe(include='all').T
missing_before = df.isna().sum()
df = analyzer.handle_missing_values(numeric_strategy='group_median', group_col='category')
df = analyzer.engineer_multimodal_features(store_arrays=False)
outliers, bounds = analyzer.detect_outliers('unit_price')
df = analyzer.cap_outliers('unit_price', output_col='unit_price_capped')
rfm = analyzer.calculate_rfm()
missing_after = df.isna().sum()
missing_before, missing_after, bounds, rfm.head()


IQR 기준 단가 이상치는 39,627건(7.31%)이다. 결측치는 136534개에서 135080개로 감소했다. 상품명 단어 수, 문자열 길이, 이미지 평균, 이미지 표준편차, 엣지 강도는 반복문이 아니라 NumPy/Pandas 벡터 연산으로 산출한다.

In [ ]:
numeric_cols = ['quantity', 'unit_price', 'amount', 'name_word_count', 'image_mean', 'image_std', 'edge_strength']
numeric_summary = analyzer.summarize_numeric(numeric_cols)
corr = analyzer.correlation_matrix(['quantity', 'unit_price_capped', 'amount', 'name_word_count', 'image_mean', 'image_std', 'edge_strength'])
numeric_summary, corr


![histogram_unit_price](../evidence/histogram_unit_price.png)

![boxplot_outlier_before_after](../evidence/boxplot_outlier_before_after.png)

![bar_rfm_segments](../evidence/bar_rfm_segments.png)

![heatmap_correlation](../evidence/heatmap_correlation.png)

![scatter_image_mean_price](../evidence/scatter_image_mean_price.png)

![line_monthly_revenue](../evidence/line_monthly_revenue.png)

![bonus_cohort_retention_heatmap](../evidence/bonus_cohort_retention_heatmap.png)

RFM 세그먼트별 고객 수, 평균 최근성, 평균 구매 빈도, 평균 구매 금액은 `evidence/rfm_segment_summary.csv`에 저장했다. 막대그래프는 세그먼트 규모를, 라인차트는 월별 매출 추세를, 히트맵은 수치 피처 간 상관관계와 코호트 유지율을 확인하는 근거로 사용한다.